# multi_strat_027

> **Auto-generated documentation header.** Fill the `TODO` fields after review.
> Source notebook: `Notebook1 (203).ipynb`

| Field | Value |
|---|---|
| Section | `strategies` |
| Market | `unknown` |
| Trading style | `unknown` |
| Timeframe | `unknown` |
| Code cells | 8 |
| Detected functions | resample_1hr_custom, compute_atr, compute_indicators, backtest_ticker, run_backtest, calculate_metrics |

### Research documentation
- **Hypothesis:** _TODO_
- **Alpha source:** _TODO_
- **Indicators / features:** _TODO_
- **Entry logic:** _TODO_
- **Exit logic (SL/TP):** _TODO_
- **Risk management:** _TODO_
- **Assumptions & limitations:** _TODO_
- **Performance (if available):** _TODO_

> Shared utilities live in `src/qresearch/` — prefer importing `jma_signals`, `backtest_trades`,
> `calculate_metrics`, `resample_ohlcv` instead of re-implementing them here.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
data = pd.read_parquet('/content/drive/MyDrive/Binance_Crypto_T5_2020_25.parquet')
data.rename(columns={
    'Datetime': 'Date',
    'Open': 'open',
    'Close': 'close',
    'Low': 'low',
    'High': 'high',
    'Volume': 'volume'
}, inplace=True)

data

In [ ]:
data['Ticker'].unique()

In [ ]:
# Convert 'Date' column to datetime if it isn't already
data['Date'] = pd.to_datetime(data['Date'])

# Define the date range
start_date = '2025-01-01'
end_date = '2025-12-31'

# Define the tickers you want to filter
selected_tickers = ['BTCUSDT', 'SOLUSDT', 'XRPUSDT', 'BNBUSDT', 'ETHUSDT']  # Add tickers you want

# Apply filtering
data = data[
    (data["Date"] >= start_date) &
    (data["Date"] <= end_date) &
    (data["Ticker"].isin(selected_tickers))
]

# Display filtered results
data

In [ ]:
import pandas as pd

# Ensure Date is datetime
data['Date'] = pd.to_datetime(data['Date'])

# Custom resampling function with offset
def resample_1hr_custom(group):
    ticker = group['Ticker'].iloc[0]
    group = group.set_index('Date')
    resampled = group.resample('30min').agg({
        'open': 'first',
        'high': 'max',
        'low': 'min',
        'close': 'last',
    }).dropna().reset_index()
    resampled['Ticker'] = ticker
    return resampled

# Apply per ticker
data_1hr = data.groupby('Ticker').apply(resample_1hr_custom).reset_index(drop=True)

# Reorder columns
data_1hr = data_1hr[['Ticker', 'Date', 'open', 'high', 'low', 'close']]

data = data_1hr
data

### **INDICATOR CODE**

In [ ]:
import numpy as np
import pandas as pd
from numba import njit
from tqdm import tqdm

# =====================================================
# MINIMAL TRADING STRATEGY
# JMA State + Fast Trigger + ATR Trailing Stop ONLY
# =====================================================

def compute_atr(df: pd.DataFrame, period: int = 14) -> pd.Series:
    """Calculate Average True Range."""
    h, l, c = df['high'], df['low'], df['close']
    tr = pd.concat([
        h - l,
        (h - c.shift(1)).abs(),
        (l - c.shift(1)).abs()
    ], axis=1).max(axis=1)
    return tr.rolling(window=period, min_periods=1).mean()


def compute_indicators(
    df: pd.DataFrame,
    smooth_length: int = 12,
    jma_norm_period: int = 60,
    atr_mult: float = 0.1,
    accel_threshold: float = 0.5
) -> pd.DataFrame:
    """Compute minimal indicators."""
    df = df.copy()
    price = df['close']

    # JMA & State
    alpha = 2.0 / (smooth_length + 1)
    df['jma'] = price.ewm(alpha=alpha, adjust=False).mean()
    df['jma_norm'] = (df['jma'] - df['jma'].rolling(jma_norm_period).mean()) / df['jma'].rolling(jma_norm_period).std()
    df['jma_state'] = df['jma_norm'].rolling(jma_norm_period).apply(lambda x: x.rank(pct=True).iloc[-1], raw=False)

    # Fast Trigger: JMA Slope Acceleration
    df['jma_slope'] = df['jma'].diff()
    df['jma_accel'] = df['jma_slope'].diff(3)
    accel_std = df['jma_accel'].rolling(60).std()
    df['jma_accel_norm'] = df['jma_accel'] / accel_std.replace(0, 1)

    # Generate Signals
    df['signal'] = 0

    # LONG: Bullish state + Positive acceleration
    df.loc[(df['jma_state'] > 0.6) & (df['jma_accel_norm'] > accel_threshold), 'signal'] = 1

    # SHORT: Bearish state + Negative acceleration
    df.loc[(df['jma_state'] < 0.4) & (df['jma_accel_norm'] < -accel_threshold), 'signal'] = -1

    # ATR Trailing Stop
    df['atr_14'] = compute_atr(df, 14)
    df['atr_5'] = compute_atr(df, 5)
    atr_14_max = df['atr_14'].rolling(20).max() * atr_mult
    atr_5_max = df['atr_5'].rolling(20).max() * atr_mult
    df['atr_trail'] = pd.concat([atr_14_max, atr_5_max], axis=1).max(axis=1)

    return df


@njit
def backtest_ticker(
    opens, highs, lows, closes, signals, atr_trails,
    capital, fee_rate, slippage_rate, initial_sl_pct
):
    """Execute backtest for single ticker."""
    n = len(opens)

    # Trade records
    entry_idx, exit_idx = [], []
    entry_prices, exit_prices = [], []
    quantities, directions = [], []
    pnls = []

    # Position state
    position_active = False
    position_direction = 0
    position_entry_price = 0.0
    position_size = 0.0
    trailing_stop = 0.0
    position_entry_idx = 0

    for i in range(n):
        if not position_active:
            # Entry logic
            if signals[i] != 0:
                direction = signals[i]
                entry_price = opens[i] * (1 + slippage_rate * direction)
                position_size = capital / entry_price
                initial_stop = opens[i] * (1 - direction * initial_sl_pct)

                # Check immediate stop-out
                stopped = (direction == 1 and lows[i] <= initial_stop) or (direction == -1 and highs[i] >= initial_stop)

                if stopped:
                    exit_price = initial_stop * (1 - slippage_rate * direction)
                    gross_pnl = direction * (exit_price - entry_price) * position_size
                    total_fees = fee_rate * (entry_price + exit_price) * position_size

                    entry_idx.append(i)
                    exit_idx.append(i)
                    entry_prices.append(entry_price)
                    exit_prices.append(exit_price)
                    quantities.append(position_size)
                    directions.append(direction)
                    pnls.append(gross_pnl - total_fees)
                    continue

                # Open position
                position_active = True
                position_entry_price = entry_price
                position_direction = direction
                trailing_stop = initial_stop
                position_entry_idx = i

        else:
            # Exit logic: Trailing stop only
            prev_close = closes[i - 1]
            effective_atr = atr_trails[i]

            if position_direction == 1:
                trailing_stop = max(trailing_stop, prev_close - effective_atr)
                hit_stop = lows[i] <= trailing_stop
            else:
                trailing_stop = min(trailing_stop, prev_close + effective_atr)
                hit_stop = highs[i] >= trailing_stop

            if hit_stop:
                exit_price = trailing_stop * (1 - slippage_rate * position_direction)
                gross_pnl = position_direction * (exit_price - position_entry_price) * position_size
                total_fees = fee_rate * (position_entry_price + exit_price) * position_size

                entry_idx.append(position_entry_idx)
                exit_idx.append(i)
                entry_prices.append(position_entry_price)
                exit_prices.append(exit_price)
                quantities.append(position_size)
                directions.append(position_direction)
                pnls.append(gross_pnl - total_fees)
                position_active = False

    return (entry_idx, exit_idx, entry_prices, exit_prices,
            quantities, directions, pnls)


def run_backtest(
    data: pd.DataFrame,
    initial_capital: float = 100000,
    fee_rate: float = 0.0003,
    slippage_rate: float = 0.0003,
    initial_sl_pct: float = 0.02
) -> pd.DataFrame:
    """Run backtest across all tickers."""
    all_trades = []

    for ticker in tqdm(data['Ticker'].unique(), desc="Backtesting"):
        ticker_data = data[data['Ticker'] == ticker].sort_values('Date').reset_index(drop=True)

        results = backtest_ticker(
            ticker_data['open'].values,
            ticker_data['high'].values,
            ticker_data['low'].values,
            ticker_data['close'].values,
            ticker_data['signal'].values,
            ticker_data['atr_trail'].values,
            initial_capital, fee_rate, slippage_rate, initial_sl_pct
        )

        dates = ticker_data['Date'].values
        for j in range(len(results[0])):
            all_trades.append({
                'Ticker': ticker,
                'Entry Date': dates[results[0][j]],
                'Exit Date': dates[results[1][j]],
                'bars_held': results[1][j] - results[0][j],
                'Entry Price': results[2][j],
                'Exit Price': results[3][j],
                'qty': results[4][j],
                'Direction': 'LONG' if results[5][j] == 1 else 'SHORT',
                'PnL': results[6][j]
            })

    if not all_trades:
        return pd.DataFrame()

    df_trades = pd.DataFrame(all_trades).sort_values(['Ticker', 'Entry Date']).reset_index(drop=True)

    return df_trades


def calculate_metrics(tradebook: pd.DataFrame, data: pd.DataFrame):
    """Calculate key performance metrics."""

    # Trades per day
    date_range = (data['Date'].max() - data['Date'].min()).days
    trades_per_day = len(tradebook) / max(date_range, 1)

    # Avg holding time (in bars)
    avg_holding_bars = tradebook['bars_held'].mean()

    # Profit factor
    winners = tradebook[tradebook['PnL'] > 0]['PnL'].sum()
    losers = abs(tradebook[tradebook['PnL'] < 0]['PnL'].sum())
    profit_factor = winners / losers if losers > 0 else float('inf')

    # Max Drawdown
    cumulative_pnl = tradebook['PnL'].cumsum()
    running_max = cumulative_pnl.cummax()
    drawdown = cumulative_pnl - running_max
    max_dd = drawdown.min()

    return {
        'trades_per_day': trades_per_day,
        'avg_holding_bars': avg_holding_bars,
        'profit_factor': profit_factor,
        'max_drawdown': max_dd
    }


# === MAIN EXECUTION ===
if __name__ == "__main__":
    # Load and prepare data
    # data = pd.read_csv("your_data.csv")
    # Uncomment the line above and comment the line below when using your own data
    # data = your_dataframe_here

    data = data.sort_values(['Ticker', 'Date']).reset_index(drop=True)

    # Compute indicators with lookahead bias prevention
    processed = []
    for ticker, group in tqdm(data.groupby('Ticker'), desc="Preparing"):
        group = compute_indicators(group)
        # Shift signals to prevent lookahead
        group['signal'] = group['signal'].shift(1)
        group['atr_trail'] = group['atr_trail'].shift(1)
        processed.append(group)

    data = pd.concat(processed, ignore_index=True).dropna()

    # Run backtest
    tradebook = run_backtest(data)
    tradebook.to_csv('/content/drive/MyDrive/tradebook_minimal.csv', index=False)

    # Calculate and display metrics
    metrics = calculate_metrics(tradebook, data)

    print("\n=== MINIMAL STRATEGY RESULTS ===")
    print(f"Trades/Day: {metrics['trades_per_day']:.3f}")
    print(f"Avg Holding Time: {metrics['avg_holding_bars']:.1f} bars")
    print(f"Profit Factor: {metrics['profit_factor']:.2f}")
    print(f"Max Drawdown: ${metrics['max_drawdown']:,.2f}")

In [ ]:
tradebook.to_csv('/content/drive/MyDrive/tradebook.csv', index=False)